# Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines

This notebook trains a Random Forest model on the Flu Shot Learning dataset to predict the probabilities of respondents receiving the H1N1 and seasonal flu vaccines.

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
print("all set")

all set


## 1. Load Data
We will load the training features, training labels, and test features from the `data/raw` folder.

In [7]:
DATA_DIR = '/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'
SUBMISSIONS_DIR = '../../submissions'

# Ensure submissions directory exists
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

train_features = pd.read_csv(os.path.join(DATA_DIR, 'training_set_features.csv'), index_col='respondent_id')
train_labels = pd.read_csv(os.path.join(DATA_DIR, 'training_set_labels.csv'), index_col='respondent_id')
test_features = pd.read_csv(os.path.join(DATA_DIR, 'test_set_features.csv'), index_col='respondent_id')

# Merge features and labels to easily check data
train_df = train_features.join(train_labels)
train_df.head()

,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,doctor_recc_h1n1,...,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine
respondent_id,,,,,,,,,,,,,,,,,,,,,
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,NaN,NaN,0,0
1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1
2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,...,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0
3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,NaN,NaN,0,1
4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0


## 2. Preprocessing
We need to handle missing values and encode categorical features. We use `SimpleImputer` and `StandardScaler` for numeric columns, and `SimpleImputer` and `OneHotEncoder` for categorical columns.

In [8]:
numeric_cols = train_features.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = train_features.select_dtypes(include=['object']).columns

# Numeric transformer: Impute missing values with median and scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical transformer: Impute missing with 'missing' and one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

## 3. Model Definition
We use a `RandomForestClassifier` wrapped in a `MultiOutputClassifier` to predict both targets (`h1n1_vaccine` and `seasonal_vaccine`) simultaneously.

In [9]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
multi_target_rf = MultiOutputClassifier(rf, n_jobs=-1)

clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', multi_target_rf)])

## 4. Training and Evaluation
We split the training data into a training set and a validation set to evaluate the model's performance using ROC AUC.

In [10]:
X_train, X_eval, y_train, y_eval = train_test_split(
    train_features,
    train_labels,
    test_size=0.2,
    random_state=42
)

# Train on training split
clf.fit(X_train, y_train)

# Predict probabilities on evaluation split
y_preds = clf.predict_proba(X_eval)

# predict_proba returns a list of arrays, one for each target.
# We extract the probabilities for the positive class (index 1)
y_preds_h1n1 = y_preds[0][:, 1]
y_preds_seasonal = y_preds[1][:, 1]

# Calculate ROC AUC
roc_auc_h1n1 = roc_auc_score(y_eval['h1n1_vaccine'], y_preds_h1n1)
roc_auc_seasonal = roc_auc_score(y_eval['seasonal_vaccine'], y_preds_seasonal)
mean_roc_auc = (roc_auc_h1n1 + roc_auc_seasonal) / 2

print(f"ROC AUC for H1N1: {roc_auc_h1n1:.4f}")
print(f"ROC AUC for Seasonal: {roc_auc_seasonal:.4f}")
print(f"Mean ROC AUC: {mean_roc_auc:.4f}")

ROC AUC for H1N1: 0.8285
ROC AUC for Seasonal: 0.8536
Mean ROC AUC: 0.8410


## 5. Full Train and Submission
Finally, we retrain the model on the entire dataset to maximize the amount of training data, and generate the submission file.

In [11]:
print("Training on full dataset...")
clf.fit(train_features, train_labels)

print("Predicting on test dataset...")
test_probas = clf.predict_proba(test_features)

# Load submission format
submission_df = pd.read_csv(os.path.join(DATA_DIR, 'submission_format.csv'), index_col='respondent_id')

# Populate with predictions
submission_df['h1n1_vaccine'] = test_probas[0][:, 1]
submission_df['seasonal_vaccine'] = test_probas[1][:, 1]

submission_path = os.path.join(SUBMISSIONS_DIR, 'submission.csv')
submission_df.to_csv(submission_path)
print(f"Submission saved to {submission_path}")

Training on full dataset...
Predicting on test dataset...
Submission saved to ../../submissions/submission.csv
